# Tutorial 2: Virus Phylogenetic Tree Monophyly Analysis

**病毒系统发育树单系性分析 -- 使用 PyiTOL 检测分类群的系统发育关系**

This notebook demonstrates how to use PyiTOL's monophyly detection module to classify
taxonomic groups as monophyletic, paraphyletic, or polyphyletic. We will:

1. Generate a synthetic 30-leaf virus phylogenetic tree with known group structure
2. Create taxonomy data with Genotype and Clade columns
3. Run `check_monophyly()` to classify each group
4. Interpret the three-way classification results
5. Generate a color-strip template with monophyly-based coloring

本教程展示如何使用 PyiTOL 的单系性检测模块将分类群分为单系群、并系群或多系群。

---

**Background / 背景知识:**

| Term | Definition |
|------|------------|
| **Monophyletic** (单系群) | All members share a common ancestor, and ALL descendants of that ancestor are included |
| **Paraphyletic** (并系群) | All members share a common ancestor, but SOME descendants are excluded |
| **Polyphyletic** (多系群) | Members are scattered across multiple independent clades |

## 1. Setup and Imports

导入所需的库。

In [ ]:
import random
import tempfile
from pathlib import Path

import dendropy
import pandas as pd

from pyitol.core.monophyly import check_monophyly
from pyitol.templates.generator import (
    generate_color_strip_template,
    generate_branch_template,
)
from pyitol.templates.presets.nature import get_nature_palette

## 2. Build a Synthetic Virus Tree with Known Monophyly Structure

We construct a tree where we **know** which groups should be monophyletic, paraphyletic,
or polyphyletic. This lets us verify the algorithm's output.

The tree has 30 leaves organized as follows:

- **Genotype A** (8 leaves): monophyletic -- all in one clade
- **Genotype B** (8 leaves): monophyletic -- all in one clade
- **Genotype C** (6 leaves): polyphyletic -- members split across two separate clades
- **Genotype D** (8 leaves): paraphyletic -- most members in one clade, but the clade also contains some Genotype C members

我们构建一棵已知单系性结构的树，以便验证算法输出。

In [ ]:
# We build the tree from a Newick string that encodes our desired topology.
# Structure:
#   root
#   ├── Clade1 (Genotype A: A_01..A_08) -- monophyletic
#   ├── Clade2
#   │   ├── SubClade2a (Genotype D: D_01..D_05)
#   │   └── SubClade2b (Genotype C: C_01..C_03)  <-- D is paraphyletic because C is nested
#   ├── Clade3 (Genotype B: B_01..B_08) -- monophyletic
#   └── Clade4
#       ├── SubClade4a (Genotype C: C_04..C_06)
#       └── SubClade4b (Genotype D: D_06..D_08)  <-- C is polyphyletic (split across Clade2 & Clade4)

newick = (
    "((((A_01:0.1,A_02:0.1):0.1,(A_03:0.1,A_04:0.1):0.1):0.1,"
    "((A_05:0.1,A_06:0.1):0.1,(A_07:0.1,A_08:0.1):0.1):0.1):0.2,"
    "(((D_01:0.1,D_02:0.1):0.1,(D_03:0.1,(D_04:0.1,D_05:0.1):0.1):0.1):0.1,"
    "(C_01:0.1,(C_02:0.1,C_03:0.1):0.1):0.2):0.2,"
    "((B_01:0.1,B_02:0.1):0.1,(B_03:0.1,(B_04:0.1,(B_05:0.1,(B_06:0.1,(B_07:0.1,B_08:0.1):0.1):0.1):0.1):0.1):0.1):0.2,"
    "((C_04:0.1,C_05:0.1):0.1,(C_06:0.1,(D_06:0.1,(D_07:0.1,D_08:0.1):0.1):0.1):0.1):0.2):0.3);"
)

tree = dendropy.Tree.get(data=newick, schema="newick", preserve_underscores=True)

tmpdir = Path(tempfile.mkdtemp(prefix="pyitol_monophyly_"))
tree_path = tmpdir / "virus_tree.nwk"
tree.write_to_path(str(tree_path), schema="newick")

leaf_names = sorted([n.taxon.label for n in tree.leaf_node_iter()])
print(f"Tree written to: {tree_path}")
print(f"Number of leaves: {len(leaf_names)}")
print(f"Leaf names: {leaf_names}")

## 3. Create Virus Taxonomy Metadata

We assign each leaf to a Genotype and Clade based on our designed topology.

根据设计的拓扑结构为每个叶节点分配 Genotype 和 Clade。

In [ ]:
# Define the genotype assignments
genotype_map = {}
for i in range(1, 9):
    genotype_map[f"A_{i:02d}"] = ("A", "Clade_I")
    genotype_map[f"B_{i:02d}"] = ("B", "Clade_III")
    genotype_map[f"D_{i:02d}"] = ("D", "Clade_II" if i <= 5 else "Clade_IV")
for i in range(1, 7):
    genotype_map[f"C_{i:02d}"] = ("C", "Clade_II" if i <= 3 else "Clade_IV")

records = []
for name in leaf_names:
    geno, clade = genotype_map[name]
    records.append({
        "id": name,
        "Genotype": geno,
        "Clade": clade,
        "Collection_Year": random.choice([2018, 2019, 2020, 2021, 2022]),
    })

taxonomy_df = pd.DataFrame(records)
taxonomy_path = tmpdir / "virus_taxonomy.csv"
taxonomy_df.to_csv(taxonomy_path, index=False)

print(f"Taxonomy saved to: {taxonomy_path}")
taxonomy_df

## 4. Run Monophyly Analysis

We use `check_monophyly()` to test each Genotype group. The function returns a list of
dictionaries with the monophyly status for each group.

使用 `check_monophyly()` 检测每个 Genotype 组的单系性。

In [ ]:
# Run monophyly check at the Genotype level
results = check_monophyly(
    taxonomy_path=str(taxonomy_path),
    tree_path=str(tree_path),
    rank="Genotype",
    id_column="id",
)

# Display as a DataFrame for clarity
results_df = pd.DataFrame(results)
print("Monophyly analysis results:")
results_df[["group", "status", "lca_node", "member_count", "extra_count", "missing_count"]]

### 4.1 Interpret the Results

Let's examine each group's classification in detail.

详细查看每个组的分类结果。

In [ ]:
for r in results:
    print(f"\n{'='*60}")
    print(f"Group: {r['group']}")
    print(f"  Status:          {r['status'].upper()}")
    print(f"  LCA node:        {r['lca_node']}")
    print(f"  Members:         {r['member_count']}")
    print(f"  Extra nodes:     {r['extra_count']} ({r['extra_nodes'][:80]}{'...' if len(r['extra_nodes']) > 80 else ''})")
    print(f"  Missing nodes:   {r['missing_count']} ({r['missing_nodes']})")
    if r['subgroups']:
        print(f"  Subgroups:       {r['subgroups'][:120]}{'...' if len(r['subgroups']) > 120 else ''}")

### 4.2 Expected Results Interpretation

Based on our tree construction:

| Group | Expected | Why |
|-------|----------|-----|
| Genotype A | **Monophyletic** | All 8 members in one clade, no outsiders |
| Genotype B | **Monophyletic** | All 8 members in one clade, no outsiders |
| Genotype C | **Polyphyletic** | Members split across Clade_II (C_01-03) and Clade_IV (C_04-06) |
| Genotype D | **Polyphyletic** | Members split across Clade_II (D_01-05) and Clade_IV (D_06-08); may appear paraphyletic depending on topology |

根据我们的树构建方式，预期结果如上表所示。

## 5. Monophyly-Based Color-Strip Template

We generate a color-strip template where:
- **Monophyletic** groups get solid, distinct colors
- **Polyphyletic** groups get a striped/pattern indicator (lighter shade)

This helps visually identify which groups need taxonomic revision.

生成基于单系性分类的颜色条模板，帮助直观识别需要分类修订的类群。

In [ ]:
# Assign colors: monophyletic groups get strong colors, polyphyletic get muted
status_color_map = {}
strong_palette = get_nature_palette("vibrant", n=10)
muted_palette = get_nature_palette("pastel", n=10)

genotype_colors = {}
color_idx = 0
for r in results:
    grp = r["group"]
    if r["status"] == "monophyletic":
        genotype_colors[grp] = strong_palette[color_idx % len(strong_palette)]
    else:
        # Polyphyletic/paraphyletic groups get a lighter shade
        genotype_colors[grp] = muted_palette[color_idx % len(muted_palette)]
    color_idx += 1

print("Genotype color mapping (monophyly-based):")
for r in results:
    grp = r["group"]
    status = r["status"]
    color = genotype_colors[grp]
    marker = "*" if status == "monophyletic" else "!"
    print(f"  {marker} {grp:12s} [{status:14s}] -> {color}")

In [ ]:
# Generate the color-strip template
mono_strip_path = tmpdir / "genotype_monophyly_strip.txt"

generate_color_strip_template(
    output_path=mono_strip_path,
    taxonomy_path=taxonomy_path,
    tree_path=tree_path,
    column="Genotype",
    colors=genotype_colors,
    label="Genotype_monophyly",
    strip_width="60",
)

print(f"Color-strip template written to: {mono_strip_path}")
print("--- Preview ---")
print(mono_strip_path.read_text()[:800])

In [ ]:
# Also generate a branch coloring template
branch_path = tmpdir / "genotype_branch_colors.txt"

generate_branch_template(
    output_path=branch_path,
    taxonomy_path=taxonomy_path,
    tree_path=tree_path,
    column="Genotype",
    colors=genotype_colors,
    label="Genotype_branches",
)

print(f"Branch template written to: {branch_path}")

## 6. Summary of Generated Files

查看所有生成的文件。

In [ ]:
all_files = sorted(tmpdir.iterdir())
print("Generated files:")
for f in all_files:
    size = f.stat().st_size
    print(f"  {f.name:40s}  {size:>6d} bytes")

## 7. Summary

In this tutorial we demonstrated PyiTOL's monophyly analysis workflow:

1. **Synthetic tree construction**: Built a 30-leaf virus tree with known group structure
2. **Taxonomy metadata**: Created Genotype and Clade columns
3. **Monophyly detection**: Used `check_monophyly()` to classify groups as monophyletic/paraphyletic/polyphyletic
4. **Result interpretation**: Examined extra nodes, missing nodes, and subgroups
5. **Monophyly-based visualization**: Generated color-strip and branch templates with colors reflecting monophyly status

**本教程总结：**

我们展示了 PyiTOL 的单系性分析工作流，包括构建已知结构的病毒树、分类元数据创建、
单系性检测（三分类：单系/并系/多系）、结果解释，以及基于单系性状态的颜色可视化。

---

### Key Takeaways / 关键要点

- **Monophyletic groups** are taxonomically coherent; all members share a common ancestor exclusive to the group
- **Polyphyletic groups** may need taxonomic revision; members are scattered across the tree
- The `subgroups` field in the results helps identify how polyphyletic groups are distributed
- Using colorblind-friendly palettes ensures accessibility in publications

---

### Next Steps / 后续步骤

- See `03_batch_workflow.ipynb` for processing multiple trees automatically

In [ ]:
# Cleanup (optional)
# import shutil; shutil.rmtree(tmpdir)
print(f"Tutorial complete. Temporary files are in: {tmpdir}")